# 🚀 KHỐI B-1: 5-FOLD CNN OUT-OF-FOLD (ÂM TIẾT HÁN NÔM)
Notebook chạy trên Kaggle GPU (T4 hoặc P100) để huấn luyện 5 fold out-of-fold và trích xuất xác suất `p_visual_oof.csv`.

**Cài đặt:** Chọn **Settings -> Accelerator -> GPU T4 x2 (hoặc GPU P100)**.

In [ ]:
!nvidia-smi

### 1. Tìm tệp dữ liệu đầu vào (`crops.npz` và `labels_final.csv`)

In [ ]:
import os, sys, glob
from pathlib import Path

# Tìm thư mục chứa crops.npz
crops_files = glob.glob("/kaggle/input/**/crops.npz", recursive=True) + glob.glob("**/crops.npz", recursive=True)
labels_files = glob.glob("/kaggle/input/**/labels_final.csv", recursive=True) + glob.glob("**/labels_final.csv", recursive=True)

assert len(crops_files) > 0, "Không tìm thấy crops.npz! Hãy kiểm tra đã add dataset vào notebook chưa."
assert len(labels_files) > 0, "Không tìm thấy labels_final.csv!"

crops_path = crops_files[0]
labels_path = labels_files[0]
print(f"✓ crops.npz: {crops_path} ({os.path.getsize(crops_path) / (1024*1024):.2f} MB)")
print(f"✓ labels_final.csv: {labels_path} ({os.path.getsize(labels_path) / (1024*1024):.2f} MB)")

### 2. Chạy huấn luyện 5-Fold CNN Out-of-fold (B-1)
Thời gian chạy ước tính trên GPU T4 / P100: **~10–12 phút** (15 epochs x 5 folds).

In [ ]:
# Tìm hoặc copy file train_oof_cnn.py
script_files = glob.glob("/kaggle/input/**/train_oof_cnn.py", recursive=True) + glob.glob("**/train_oof_cnn.py", recursive=True)
if script_files:
    script_path = script_files[0]
else:
    # Fallback nếu tải notebook riêng
    script_path = "train_oof_cnn.py"

!python {script_path} --crops "{crops_path}" --labels "{labels_path}" --out-dir /kaggle/working/output --epochs 15 --bs 256

### 3. Kiểm tra kết quả đầu ra (`p_visual_oof.csv`)

In [ ]:
import pandas as pd
import json

out_csv = "/kaggle/working/output/p_visual_oof.csv"
summary_file = "/kaggle/working/output/summary_oof.json"

assert os.path.exists(out_csv), "Không thấy file kết quả p_visual_oof.csv!"
df_res = pd.read_csv(out_csv)
print(f"✓ p_visual_oof.csv có {len(df_res):,} dòng ({os.path.getsize(out_csv) / (1024*1024):.2f} MB).")
print("\n--- 5 DÒNG ĐẦU TIÊN ---")
display(df_res.head())

print("\n--- PHÂN BỐ XÁC SUẤT p_visual ---")
print(df_res["p_visual"].describe())

if os.path.exists(summary_file):
    print("\n--- TỔNG KẾT METRICS (summary_oof.json) ---")
    with open(summary_file) as f:
        print(json.dumps(json.load(f), indent=2))

### 4. Đóng gói kết quả & Tải về máy Mac

In [ ]:
!cd /kaggle/working/output && zip -r /kaggle/working/p_visual_oof_results.zip p_visual_oof.csv summary_oof.json classes.json
print("\n✅ ĐÃ TẠO FILE NÉN: /kaggle/working/p_visual_oof_results.zip")
from IPython.display import FileLink, display, HTML
display(HTML('<h3>📥 Bấm trực tiếp vào link dưới đây để tải về máy:</h3>'))
display(FileLink('p_visual_oof_results.zip'))